# 🛡️ BOI Sentinel AI — XGBoost Risk Scoring Model Training (v4)

**Purpose:** Train an XGBoost binary classifier for Android APK malware detection.  
**Output:** `xgb_risk_model.json` + `drebin_feature_names.json` → copy to `services/risk-scoring/models/`

---

### 📦 Dataset Links

| Dataset | Samples | Classes | Link |
|:---|:---|:---|:---|
| **DREBIN-215** (Primary) | 15,036 | Binary (Benign/Malware) | [Kaggle](https://www.kaggle.com/datasets/shashwatwork/android-malware-dataset-for-machine-learning) |
| **CICMalDroid-2020** (Multi-class) | 17,341 | 5-class (Adware/Banking/SMS/Riskware/Benign) | [Kaggle](https://www.kaggle.com/datasets/subhajournal/cicmaldroid2020) |
| **Android Permissions Dataset** | 29,332 | Binary | [Kaggle](https://www.kaggle.com/datasets/saurabhshahane/android-permission-dataset) |
| **CCCS-CIC-AndMal-2020** | 200K+ | Multi-family | [UNB CIC](https://www.unb.ca/cic/datasets/andmal2020.html) |

This notebook uses **DREBIN-215** (auto-downloaded via kagglehub).

### 📋 Model Format: `.json` (Recommended)
- Cross-platform & cross-version compatible
- No pickle security risks
- Already used by `model.py` → drop-in replacement

## 1. Install Dependencies

In [ ]:
!pip install -q kagglehub xgboost==2.0.3 shap==0.45.1 scikit-learn pandas numpy matplotlib seaborn

## 2. Imports

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import kagglehub
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score, f1_score,
    roc_curve, precision_recall_curve, average_precision_score
)

print(f'XGBoost version: {xgb.__version__}')
print(f'SHAP version: {shap.__version__}')

## 3. Download DREBIN-215 Dataset from Kaggle

In [ ]:
# Auto-download from Kaggle (requires kaggle auth — set KAGGLE_USERNAME & KAGGLE_KEY)
path = kagglehub.dataset_download('shashwatwork/android-malware-dataset-for-machine-learning')
print(f'Dataset downloaded to: {path}')

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'CSV files found: {csv_files}')

CSV_PATH = os.path.join(path, csv_files[0])
print(f'Using: {CSV_PATH}')

## 4. Load & Explore Data

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {len(df.columns)} ({len(df.columns)-1} features + 1 label)')
print(f'\n--- Class Distribution ---')
label_col = df.columns[-1]
print(df[label_col].value_counts())
print(f'\nLabel column: "{label_col}"')
print(f'\n--- Sample Feature Names (first 20) ---')
print(list(df.columns[:20]))
print(f'\n--- Data Types ---')
print(df.dtypes.value_counts())
print(f'\n--- Missing Values ---')
print(f'Total NaN: {df.isna().sum().sum()}')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df[label_col].value_counts()
labels_map = {0: 'Benign', 1: 'Malware'}
colors = ['#22c55e', '#ef4444']

axes[0].bar([labels_map.get(i, str(i)) for i in counts.index], counts.values, color=colors)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Feature activation heatmap (sample)
sample = df.iloc[:100, :50]
axes[1].imshow(sample.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
axes[1].set_title('Feature Activation (first 100 samples x 50 features)')
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Samples')

plt.tight_layout()
plt.show()

## 5. Prepare Features & Labels

In [ ]:
# Separate features and labels
X = df.iloc[:, :-1].copy()
y = df.iloc[:, -1].copy()

# Clean feature names (remove special chars that XGBoost doesn't like)
feature_names = []
for col in X.columns:
    clean = str(col).strip()
    feature_names.append(clean)
X.columns = feature_names

# Handle any NaN values
X = X.fillna(0)

# Ensure binary values
X = X.astype(np.float32)
y = y.astype(int)

# Class imbalance ratio
n_benign = (y == 0).sum()
n_malware = (y == 1).sum()
scale_pos_weight = n_benign / n_malware

print(f'Features: {X.shape[1]}')
print(f'Samples:  {X.shape[0]}')
print(f'Benign:   {n_benign} ({n_benign/len(y)*100:.1f}%)')
print(f'Malware:  {n_malware} ({n_malware/len(y)*100:.1f}%)')
print(f'scale_pos_weight: {scale_pos_weight:.3f}')
print(f'\nFeature names saved: {len(feature_names)} features')

## 6. Train XGBoost Model (Tuned Hyperparameters)

In [ ]:
# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples (Benign: {(y_train==0).sum()}, Malware: {(y_train==1).sum()})')
print(f'Test:  {X_test.shape[0]} samples (Benign: {(y_test==0).sum()}, Malware: {(y_test==1).sum()})')

# Create DMatrix objects
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dtest  = xgb.DMatrix(X_test, label=y_test, feature_names=feature_names)

# ──────────────────────────────────────────────────────────────────────
# Tuned hyperparameters for DREBIN-215 APK malware classification
# ──────────────────────────────────────────────────────────────────────
params = {
    'objective':         'binary:logistic',
    'eval_metric':       ['logloss', 'auc', 'error'],
    'max_depth':         6,            # deeper trees for 215 sparse binary features
    'learning_rate':     0.05,         # slower LR + more rounds = better generalization
    'subsample':         0.8,          # row sampling to prevent overfitting
    'colsample_bytree':  0.8,          # feature sampling per tree
    'min_child_weight':  3,            # minimum samples per leaf
    'gamma':             0.1,          # minimum loss reduction for split
    'reg_alpha':         0.1,          # L1 regularization
    'reg_lambda':        1.0,          # L2 regularization
    'scale_pos_weight':  scale_pos_weight,  # handle class imbalance
    'seed':              42,
    'verbosity':         1,
}

print('\n--- Training XGBoost ---')
print(f'Params: {json.dumps(params, indent=2)}')

model = xgb.train(
    params,
    dtrain,
    num_boost_round=500,               # max rounds (early stopping will find optimal)
    evals=[(dtrain, 'train'), (dtest, 'eval')],
    early_stopping_rounds=30,          # stop if no improvement for 30 rounds
    verbose_eval=50,                   # print every 50 rounds
)

print(f'\n✅ Training complete!')
print(f'Best iteration: {model.best_iteration}')
print(f'Best eval score: {model.best_score:.6f}')

## 7. Evaluate Model

In [ ]:
# Predict on test set
y_pred_proba = model.predict(dtest)
y_pred = (y_pred_proba >= 0.5).astype(int)

# Metrics
acc  = accuracy_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_proba)
f1   = f1_score(y_test, y_pred)
ap   = average_precision_score(y_test, y_pred_proba)

print('=' * 50)
print('          MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'Accuracy:           {acc:.4f}')
print(f'ROC AUC:            {auc:.4f}')
print(f'F1 Score:           {f1:.4f}')
print(f'Avg Precision (AP): {ap:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malware']))

In [ ]:
# Visualization: Confusion Matrix + ROC Curve + Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign', 'Malware'], yticklabels=['Benign', 'Malware'])
axes[0].set_title(f'Confusion Matrix (Acc={acc:.3f})')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[2].plot(rec, prec, 'g-', linewidth=2, label=f'AP = {ap:.4f}')
axes[2].set_title('Precision-Recall Curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='lower left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 5-Fold Stratified Cross-Validation

In [ ]:
print('--- 5-Fold Stratified Cross-Validation ---\n')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc_scores = []
cv_acc_scores = []
cv_f1_scores  = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    dtrain_cv = xgb.DMatrix(X.iloc[train_idx], label=y.iloc[train_idx], feature_names=feature_names)
    dval_cv   = xgb.DMatrix(X.iloc[val_idx],   label=y.iloc[val_idx],   feature_names=feature_names)

    m = xgb.train(
        params, dtrain_cv,
        num_boost_round=model.best_iteration + 1,
        evals=[(dval_cv, 'val')],
        verbose_eval=False,
    )

    pred = m.predict(dval_cv)
    pred_label = (pred >= 0.5).astype(int)

    fold_auc = roc_auc_score(y.iloc[val_idx], pred)
    fold_acc = accuracy_score(y.iloc[val_idx], pred_label)
    fold_f1  = f1_score(y.iloc[val_idx], pred_label)

    cv_auc_scores.append(fold_auc)
    cv_acc_scores.append(fold_acc)
    cv_f1_scores.append(fold_f1)

    print(f'Fold {fold+1}: AUC={fold_auc:.4f}  Acc={fold_acc:.4f}  F1={fold_f1:.4f}')

print(f'\n--- Cross-Validation Summary ---')
print(f'AUC:      {np.mean(cv_auc_scores):.4f} \u00b1 {np.std(cv_auc_scores):.4f}')
print(f'Accuracy: {np.mean(cv_acc_scores):.4f} \u00b1 {np.std(cv_acc_scores):.4f}')
print(f'F1 Score: {np.mean(cv_f1_scores):.4f} \u00b1 {np.std(cv_f1_scores):.4f}')

## 9. SHAP Explainability Analysis

In [ ]:
# SHAP TreeExplainer for XGBoost
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Features: {len(feature_names)}')

In [ ]:
# SHAP Summary Plot — Top 25 most important features
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                  max_display=25, show=False)
plt.title('SHAP Feature Importance (Top 25)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot — Global feature importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                  max_display=20, show=False, plot_type='bar')
plt.title('SHAP Global Feature Importance (Top 20)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Print top 20 features by mean |SHAP value|
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = sorted(
    zip(feature_names, mean_shap),
    key=lambda x: x[1], reverse=True
)

print('--- Top 20 Features by Mean |SHAP Value| ---\n')
for i, (name, val) in enumerate(feature_importance[:20]):
    print(f'{i+1:3d}. {name:45s}  SHAP={val:.4f}')

## 10. Export Model (.json) + Feature Names

**After running this cell**, copy the 2 files to:
```
services/risk-scoring/models/xgb_risk_model.json
services/risk-scoring/models/drebin_feature_names.json
```

In [ ]:
OUTPUT_DIR = 'trained_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Save model as JSON (recommended format) ──────────────────────
model_path = os.path.join(OUTPUT_DIR, 'xgb_risk_model.json')
model.save_model(model_path)
model_size = os.path.getsize(model_path)
print(f'\u2705 Model saved:  {model_path} ({model_size / 1024:.1f} KB)')

# ── 2. Save feature names JSON ──────────────────────────────────────
feature_meta = {
    'feature_names': feature_names,
    'feature_count': len(feature_names),
    'model_type':    'binary:logistic',
    'best_iteration': model.best_iteration,
    'dataset':       'DREBIN-215',
    'train_samples': int(X_train.shape[0]),
    'test_samples':  int(X_test.shape[0]),
    'test_auc':      round(float(auc), 4),
    'test_accuracy':  round(float(acc), 4),
    'test_f1':       round(float(f1), 4),
    'cv_auc_mean':   round(float(np.mean(cv_auc_scores)), 4),
    'cv_auc_std':    round(float(np.std(cv_auc_scores)), 4),
}

feature_path = os.path.join(OUTPUT_DIR, 'drebin_feature_names.json')
with open(feature_path, 'w') as f:
    json.dump(feature_meta, f, indent=2)
print(f'\u2705 Feature names: {feature_path}')

# ── 3. Verify by reloading ──────────────────────────────────────────
verify_model = xgb.Booster()
verify_model.load_model(model_path)
verify_pred = verify_model.predict(dtest)
verify_auc = roc_auc_score(y_test, verify_pred)
print(f'\u2705 Verification: Reloaded model AUC = {verify_auc:.4f} (matches: {abs(verify_auc - auc) < 1e-6})')

# ── Summary ─────────────────────────────────────────────────────────
print(f'\n{"=" * 55}')
print(f'  MODEL TRAINING COMPLETE — READY FOR DEPLOYMENT')
print(f'{"=" * 55}')
print(f'  Algorithm:       XGBoost (binary:logistic)')
print(f'  Format:          .json (portable, version-safe)')
print(f'  Features:        {len(feature_names)} (DREBIN-215)')
print(f'  Best iteration:  {model.best_iteration}')
print(f'  Test AUC:        {auc:.4f}')
print(f'  Test Accuracy:   {acc:.4f}')
print(f'  Test F1:         {f1:.4f}')
print(f'  CV AUC:          {np.mean(cv_auc_scores):.4f} \u00b1 {np.std(cv_auc_scores):.4f}')
print(f'  Model size:      {model_size / 1024:.1f} KB')
print(f'{"=" * 55}')
print(f'\n\u27a1\ufe0f  Copy these files to services/risk-scoring/models/:')
print(f'    1. {model_path}')
print(f'    2. {feature_path}')

In [ ]:
# Optional: Download files from Colab
try:
    from google.colab import files
    files.download(model_path)
    files.download(feature_path)
    print('\u2705 Files downloaded!')
except ImportError:
    print('Not running in Colab — files are saved locally in:', OUTPUT_DIR)